**Title:** Beyond the Box: Milestone Four Improvement/Iteration Run  --- V2 NEW MODEL USAGE
**Author:** William Anderson  
**Date:** 8 August 2026  
**Description:** Trade-focused fine-tuning improvement using Google Colab GPU

#### Why I'm Doing One More Training Run

Milestone 3 technically worked, but the holdout testing also made the weak spot extremely obvious: Qwen got much better with the structured game prompts while the trade prompts were still pretty rough. Rather than just giving the same dataset a few more epochs and hoping for the best, I'm doing one targeted improvement run before building the Streamlit app - I could just ignore this step and submit an app - but I don't feel like half assing my final project for this Master's program.

The original training file had 72 real game examples but only 12 trade examples; this iteration keeps those examples and adds more vague, completed, and hypothetical trade examples, with a bigger emphasis on preserving player names and packages, asking for missing information, and not turning a potential trade into fake breaking news. Basically, this is me using the Milestone 3 evaluation for what it was supposed to be used for instead of ignoring the bad parts while still continuing to tell the story of improvement and iteration of this project over time. Also, this is an additional iteration on top of the Qwen OG (.5) model to instead use a better one for hopes of better results.

In [1]:
%pip install -q -U "trl==1.6.0" "transformers==5.9.0" datasets accelerate bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.0 MB/s eta 0:00:00


In [2]:
from pathlib import Path
import json
import random
import sqlite3
import time
import re
from difflib import get_close_matches

import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import SFTConfig, SFTTrainer

from google.colab import files

SEED = 42 #42 is also the answer to life
set_seed(SEED)
random.seed(SEED)

PROJECT_DIR = Path("/content")
BASE_DATASET_PATH = PROJECT_DIR / "nba_narrative_sft_dataset.jsonl"
EXPANDED_DATASET_PATH = PROJECT_DIR / "nba_narrative_sft_dataset_v3.jsonl"

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = PROJECT_DIR / "qwen-nba-1.5b-full-sft-v1"
METRICS_PATH = PROJECT_DIR / "nba_1.5b_full_sft_metrics.json"
HOLDOUT_OUTPUT_PATH = PROJECT_DIR / "nba_1.5b_holdout_outputs.json"

if not BASE_DATASET_PATH.exists():
    print("Upload nba_narrative_sft_dataset.jsonl")
    uploaded = files.upload()

assert BASE_DATASET_PATH.exists(), (
    "nba_narrative_sft_dataset.jsonl was not uploaded."
)

assert torch.cuda.is_available(), (
    "GPU not available. In Colab, choose Runtime > Change runtime type > GPU."
)

DEVICE = torch.device("cuda")
BF16_SUPPORTED = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if BF16_SUPPORTED else torch.float16

GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GB = (
    torch.cuda.get_device_properties(0).total_memory
    / (1024 ** 3)
)

#Use standard AdamW when Colab provides enough VRAM.
#Use paged 8-bit AdamW on smaller free-tier GPUs such as a T4.
OPTIMIZER = (
    "adamw_torch_fused"
    if GPU_MEMORY_GB >= 20
    else "paged_adamw_8bit"
)

print("GPU:", GPU_NAME)
print("GPU memory:", round(GPU_MEMORY_GB, 2), "GB")
print("Optimizer:", OPTIMIZER)
print("Base dataset:", BASE_DATASET_PATH)
print("New model output:", OUTPUT_DIR)


Upload nba_narrative_sft_dataset.jsonl


Saving nba_narrative_sft_dataset.jsonl to nba_narrative_sft_dataset.jsonl
GPU: Tesla T4
GPU memory: 14.56 GB
Optimizer: paged_adamw_8bit
Base dataset: /content/nba_narrative_sft_dataset.jsonl
New model output: /content/qwen-nba-1.5b-full-sft-v1


##### Updated System Instructions

I'm keeping the same overall guardrails from Milestone 3, but I'm making the trade rules a little more explicit. Some of the fine-tuned replies technically sounded cautious while still inventing a player, team, or package, which obviously defeats the entire point... while also being frustrating. Also, all this commentary is prob unneeded and is like talking to myself, but whatever i guess. Again, this updated system instruction was explicitly created with OpenAI's Chat GPT 5.6 Sol (not 5.5 this time).

In [3]:
SYSTEM_PROMPT = (
    "You are an NBA narrative writing assistant for a public-facing sports analysis app. "
    "Use only the information supplied by the user or explicitly retrieved by the application. "
    "Don't use outside knowledge, even if you believe you know it. Don't invent players, teams, "
    "statistics, injuries, rumors, betting lines, salary-cap details, contract details, roster "
    "history, draft picks, or transactions. Preserve player names, team names, scores, and trade "
    "assets exactly as supplied. If a trade question is vague and the teams or package are missing, "
    "say that the information is missing instead of creating a destination or package. When a "
    "complete trade package is supplied, discuss every team involved and give tentative grades. "
    "When a potential trade is proposed, clearly label it as hypothetical and never present it as "
    "completed news. For game statistics, don't claim that a statistic favored the winner when the "
    "supplied numbers show the opposite. Write clearly and conversationally, using natural "
    "contractions where appropriate. Basically, don't hallucinate and don't rewrite the facts."
)

print(SYSTEM_PROMPT)

You are an NBA narrative writing assistant for a public-facing sports analysis app. Use only the information supplied by the user or explicitly retrieved by the application. Don't use outside knowledge, even if you believe you know it. Don't invent players, teams, statistics, injuries, rumors, betting lines, salary-cap details, contract details, roster history, draft picks, or transactions. Preserve player names, team names, scores, and trade assets exactly as supplied. If a trade question is vague and the teams or package are missing, say that the information is missing instead of creating a destination or package. When a complete trade package is supplied, discuss every team involved and give tentative grades. When a potential trade is proposed, clearly label it as hypothetical and never present it as completed news. For game statistics, don't claim that a statistic favored the winner when the supplied numbers show the opposite. Write clearly and conversationally, using natural contr

#### Loading the Milestone 3 Dataset

I'm starting from the dataset I already built instead of recreating the 72 game examples again. That also keeps this iteration focused on the thing I'm actually trying to improve: trades.

In [4]:
base_examples = []

with BASE_DATASET_PATH.open("r", encoding="utf-8") as file:
    for line in file:
        line = line.strip()

        if line:
            base_examples.append(
                json.loads(line)
            )

print("Milestone 3 examples:", len(base_examples))

base_category_counts = (
    pd.Series(
        [
            example["category"]
            for example in base_examples
        ]
    )
    .value_counts()
    .rename_axis("category")
    .reset_index(name="count")
)

display(base_category_counts)

Milestone 3 examples: 84


,category,count
0,game_structured,54
1,game_score_only,18
2,trade_vague,4
3,trade_structured,4
4,trade_potential,4


#### Adding More Trade Examples

I'm adding 36 trade examples: 12 vague questions, 12 completed-trade examples, and 12 hypothetical/potential trades. The completed-trade section uses the same four real packages I already reviewed in Milestone 3, but with different user wording so Qwen sees more than one exact phrasing. The potential trades use real NBA names and teams, but they're explicitly hypothetical by design, which was actually what i wanted to accomplish from the start of this (or one of the goals, I should say).

The goal isn't to teach Qwen every transaction in NBA history - instead, it's to make the response behavior less awful and less like the GAI took acid when a user asks about a trade - basically, sound coherent and take make stuff up.

In [5]:
def make_example(category, user_text, assistant_text):
    return {
        "category": category,
        "prompt": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": user_text,
            },
        ],
        "completion": [
            {
                "role": "assistant",
                "content": assistant_text,
            }
        ],
    }


extra_trade_examples = []

#Vague trade examples
vague_trade_players = [
    "Ja Morant",
    "Jaylen Brown",
    "Kevin Durant",
    "Giannis Antetokounmpo",
    "Anthony Davis",
    "Luka Doncic",
    "Stephen Curry",
    "Devin Booker",
    "Trae Young",
    "Donovan Mitchell",
    "Zion Williamson",
    "Jayson Tatum",
]

vague_templates = [
    "I saw {player} got traded. Where did he go and who won the deal?",
    "What happened with the {player} trade? Grade both teams.",
    "{player} got moved? What was the package and which team won?",
]

for index, player in enumerate(vague_trade_players):
    user_text = vague_templates[
        index % len(vague_templates)
    ].format(player=player)

    assistant_text = (
        f"I don't have enough information to say where {player} was traded or grade the deal. "
        "Please provide the teams involved and the full package. I won't invent a destination, "
        "players, or draft picks that weren't supplied."
    )

    extra_trade_examples.append(
        make_example(
            "trade_vague",
            user_text,
            assistant_text,
        )
    )


#Completed trade examples
completed_trades = [
    {
        "label": "Paul George",
        "team_a": "Clippers",
        "team_b": "Thunder",
        "package": (
            "The Clippers acquired Paul George from the Thunder for Shai Gilgeous-Alexander, "
            "Danilo Gallinari, five first-round picks, and two pick swaps."
        ),
        "context": (
            "The Clippers were trying to contend immediately, while Oklahoma City was resetting."
        ),
        "reply": (
            "Clippers: B. They got the star they wanted for a win-now push, but the supplied price "
            "was enormous. Thunder: A. Oklahoma City received Shai Gilgeous-Alexander, Danilo "
            "Gallinari, five first-round picks, and two swaps. Based only on the supplied package "
            "and team directions, the Thunder get the stronger tentative grade."
        ),
    },
    {
        "label": "Jrue Holiday",
        "team_a": "Boston",
        "team_b": "Portland",
        "package": (
            "Boston acquired Jrue Holiday from Portland for Malcolm Brogdon, Robert Williams III, "
            "and two first-round picks."
        ),
        "context": (
            "Boston was pushing to contend, while Portland was rebuilding."
        ),
        "reply": (
            "Boston: A-. Boston added the win-now guard described in the package. Portland: B+. "
            "Portland received Malcolm Brogdon, Robert Williams III, and two first-round picks for "
            "a rebuilding direction. Those grades are tentative and based only on the supplied deal."
        ),
    },
    {
        "label": "Kyrie Irving",
        "team_a": "Dallas",
        "team_b": "Brooklyn",
        "package": (
            "Dallas acquired Kyrie Irving and Markieff Morris from Brooklyn for Spencer Dinwiddie, "
            "Dorian Finney-Smith, one first-round pick, and two second-round picks."
        ),
        "context": (
            "Dallas wanted more creation, while Brooklyn was moving away from the previous roster."
        ),
        "reply": (
            "Dallas: B+. Dallas received Kyrie Irving and Markieff Morris and added the creation "
            "described in the prompt. Brooklyn: B. Brooklyn received Spencer Dinwiddie, Dorian "
            "Finney-Smith, one first-round pick, and two second-round picks. The grades are tentative "
            "because I'm evaluating only the supplied package and direction."
        ),
    },
    {
        "label": "OG Anunoby",
        "team_a": "New York",
        "team_b": "Toronto",
        "package": (
            "New York acquired OG Anunoby, Precious Achiuwa, and Malachi Flynn from Toronto for "
            "RJ Barrett, Immanuel Quickley, and a second-round pick."
        ),
        "context": (
            "New York wanted more defense, while Toronto wanted younger pieces."
        ),
        "reply": (
            "New York: A-. The supplied package adds OG Anunoby, Precious Achiuwa, and Malachi Flynn "
            "to a team looking for more defense. Toronto: A-. Toronto receives RJ Barrett, Immanuel "
            "Quickley, and a second-round pick for the younger direction described. This looks "
            "balanced based only on the information provided."
        ),
    },
]

completed_templates = [
    "{package} {context} Grade both teams and don't add anything that isn't in the package.",
    "Completed trade: {package} {context} Who did better? Give each side a tentative grade.",
    "{package} {context} Give me a short trade reaction for every team involved.",
]

for trade in completed_trades:
    for template in completed_templates:
        user_text = template.format(
            package=trade["package"],
            context=trade["context"],
        )

        extra_trade_examples.append(
            make_example(
                "trade_structured",
                user_text,
                trade["reply"],
            )
        )


#Hypothetical trade examples
potential_trades = [
    (
        "Oklahoma City receives Anthony Davis from the Lakers for one young starter and three first-round picks.",
        "Oklahoma City: B. The Thunder get Anthony Davis in the supplied hypothetical, but give up a young starter and three first-round picks. Lakers: B+. Los Angeles gets a young starter and three firsts in exchange for Anthony Davis. This is only a hypothetical, and the grades could change depending on the identity of the young starter and the value of the picks."
    ),
    (
        "San Antonio receives Trae Young from Atlanta for one young starter and three first-round picks.",
        "San Antonio: B. The Spurs get Trae Young but give up a young starter and three first-round picks. Atlanta: B+. The Hawks receive the young starter and three firsts. This is a hypothetical only, and the exact grade depends heavily on the unnamed starter and pick quality."
    ),
    (
        "Miami receives Kevin Durant from Phoenix for two rotation players and two first-round picks.",
        "Miami: B+. The Heat get Kevin Durant in the supplied hypothetical for two rotation players and two first-round picks. Phoenix: C+. The Suns receive the two rotation players and two firsts. This remains hypothetical, and the return could look different once the unnamed players and pick details are known."
    ),
    (
        "New York receives Devin Booker from Phoenix for one young starter and four first-round picks.",
        "New York: B. The Knicks get Devin Booker but give up a young starter and four first-round picks. Phoenix: A-. The Suns receive the young starter and four firsts. This is only a hypothetical, and the exact value depends on the unnamed starter and protections or timing of the picks."
    ),
    (
        "Orlando receives Donovan Mitchell from Cleveland for one young starter and three first-round picks.",
        "Orlando: B. The Magic get Donovan Mitchell while giving up a young starter and three first-round picks. Cleveland: B+. The Cavaliers receive the young starter and three firsts. This is a hypothetical only, and the unnamed starter and pick details could move either grade."
    ),
    (
        "Houston receives Jaylen Brown from Boston for two young players and three first-round picks.",
        "Houston: B. The Rockets get Jaylen Brown for two young players and three first-round picks. Boston: B+. The Celtics receive the two young players and three firsts in the supplied scenario. This is hypothetical, and the grades are tentative because the unnamed players and picks matter."
    ),
    (
        "Brooklyn receives Zion Williamson from New Orleans for one young starter and two first-round picks.",
        "Brooklyn: B. The Nets get Zion Williamson while giving up a young starter and two first-round picks. New Orleans: B. The Pelicans receive the young starter and two firsts. This is a hypothetical only, and the missing identities and pick details keep both grades tentative."
    ),
    (
        "Golden State receives Giannis Antetokounmpo from Milwaukee for two young starters and four first-round picks.",
        "Golden State: B+. The Warriors get Giannis Antetokounmpo but pay two young starters and four first-round picks. Milwaukee: A-. The Bucks receive the two young starters and four firsts. This is strictly hypothetical, and the unnamed players and pick details could change the grades."
    ),
    (
        "Chicago receives Trae Young from Atlanta for one young starter and two first-round picks.",
        "Chicago: B. The Bulls get Trae Young for one young starter and two first-round picks. Atlanta: B. The Hawks receive the young starter and two firsts. This is a hypothetical only, and the unknown starter and pick details are important context."
    ),
    (
        "Sacramento receives Kevin Durant from Phoenix for two rotation players and three first-round picks.",
        "Sacramento: B+. The Kings get Kevin Durant in exchange for two rotation players and three first-round picks. Phoenix: B. The Suns receive the two rotation players and three firsts. This is hypothetical, and the unnamed players and pick quality could change the evaluation."
    ),
    (
        "Indiana receives Devin Booker from Phoenix for one young starter and three first-round picks.",
        "Indiana: B. The Pacers get Devin Booker while sending one young starter and three first-round picks. Phoenix: B+. The Suns receive that young starter and three firsts. This is only a hypothetical, and the exact assets would determine whether either grade moves."
    ),
    (
        "Memphis receives Anthony Davis from the Lakers for two young players and two first-round picks.",
        "Memphis: B+. The Grizzlies get Anthony Davis for two young players and two first-round picks. Lakers: B. Los Angeles receives the two young players and two firsts. This is a hypothetical only, and the unnamed players and pick details make the grades tentative."
    ),
]

potential_templates = [
    "Potential trade idea: {deal} Treat this only as a hypothetical and grade both teams.",
    "Hypothetically, {deal} Does this make sense for both teams? Preserve the package exactly.",
]

for index, (deal, reply) in enumerate(potential_trades):
    template = potential_templates[
        index % len(potential_templates)
    ]

    extra_trade_examples.append(
        make_example(
            "trade_potential",
            template.format(deal=deal),
            reply,
        )
    )

print("New trade sampy samples:", len(extra_trade_examples))

extra_counts = (
    pd.Series(
        [
            example["category"]
            for example in extra_trade_examples
        ]
    )
    .value_counts()
    .rename_axis("category")
    .reset_index(name="count")
)

display(extra_counts)

New trade sampy samples: 36


,category,count
0,trade_vague,12
1,trade_structured,12
2,trade_potential,12


##### Expanded Dataset Review

The target here is 120 examples total: the original 84 plus 36 additional trade examples. That leaves the real game section exactly where it was while bringing the trade section from 12 examples to 48 - unfortunately, it still isn't a huge dataset by any means, but it's a much less ridiculous game-to-trade balance than before.

In [6]:
expanded_examples = (
    base_examples
    + extra_trade_examples
)

user_prompts = [
    next(
        message["content"]
        for message in example["prompt"]
        if message["role"] == "user"
    )
    for example in expanded_examples
]

assert len(user_prompts) == len(set(user_prompts)), (
    "Duplicate user prompts found."
)

expanded_counts = (
    pd.Series(
        [
            example["category"]
            for example in expanded_examples
        ]
    )
    .value_counts()
    .rename_axis("category")
    .reset_index(name="count")
)

display(expanded_counts)

with EXPANDED_DATASET_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    for example in expanded_examples:
        file.write(
            json.dumps(
                example,
                ensure_ascii=False,
            )
            + "\n"
        )

print("Expanded examples:", len(expanded_examples))
print("Saved to:", EXPANDED_DATASET_PATH)

,category,count
0,game_structured,54
1,game_score_only,18
2,trade_vague,16
3,trade_structured,16
4,trade_potential,16


Expanded examples: 120
Saved to: /content/nba_narrative_sft_dataset_v3.jsonl


#### Train and Evaluation Split

I'm using the same stratified idea as before so every category stays represented; the model itself is loaded fresh from the original Qwen base model - I'm not stacking three more epochs on top of the Milestone 3 checkpoint because I want this to be a clean retraining run with the improved dataset, thought stacking more epochs was tempting just to see if the end result would change. probably not, but still.

In [7]:
rng = random.Random(SEED)

train_records = []
eval_records = []

categories = sorted(
    set(
        example["category"]
        for example in expanded_examples
    )
)

for category in categories:
    category_records = [
        example
        for example in expanded_examples
        if example["category"] == category
    ]

    rng.shuffle(category_records)

    eval_count = max(
        1,
        round(
            len(category_records) * 0.10
        ),
    )

    eval_records.extend(
        category_records[:eval_count]
    )

    train_records.extend(
        category_records[eval_count:]
    )

rng.shuffle(train_records)
rng.shuffle(eval_records)

train_dataset = Dataset.from_list(
    train_records
)

eval_dataset = Dataset.from_list(
    eval_records
)

print("Training examples:", len(train_dataset))
print("Evaluation examples:", len(eval_dataset))

display(
    pd.DataFrame(
        {
            "split": ["train", "evaluation"],
            "count": [
                len(train_dataset),
                len(eval_dataset),
            ],
        }
    )
)

Training examples: 107
Evaluation examples: 13


,split,count
0,train,107
1,evaluation,13


#### Model Switch - 0.5B to 1.5B

After the first iteration, the extra trade training clearly helped with some of the trade prompts, but the 0.5B model also started applying the same missing-context response to game and offseason prompts. Sooooo, rather than continuing to tweak that smaller model, I'm switching this run to `Qwen/Qwen2.5-1.5B-Instruct`. I'm keeping the expanded dataset, training setup, and same 10 holdouts as close as possible so the main new difference is model size/capacity and I can compare the results directly - basically, the only net change should be the model, but there is an potential issue for running out of memory.


###### Google Colab Training Setup

I moved the 1.5B run to Google Colab after the local Intel XPU repeatedly ran out of resources (basically, hardware limitations -  maybe i need to buy a 5K GPU). Anyways, this version removes the frequent training-checkpoint saves that were added only as protection against the local crashes. Colab still uses gradient checkpointing because that reduces training memory rather than saving recovery checkpoints (which is something I tried in a local env, but it didn't work).


In [8]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=DTYPE,
    low_cpu_mem_usage=True,
).to(DEVICE)

for parameter in model.parameters():
    parameter.requires_grad = True

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

trainable_percentage = (
    100
    * trainable_parameters
    / total_parameters
)

print("Total parameters:", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")
print("Trainable percentage:", f"{trainable_percentage:.2f}%")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Total parameters: 1,543,714,304
Trainable parameters: 1,543,714,304
Trainable percentage: 100.00%


##### Token-Length Check

I kept the 512-token limit because it worked cleanly in Milestone 3; I'm checking again only because the new trade examples repeat more of the supplied package, and truncating those details would be a pretty dumb/not smart way to fix package preservation.

In [9]:
MAX_SEQUENCE_LENGTH = 512


def full_conversation_token_length(example):
    conversation = (
        example["prompt"]
        + example["completion"]
    )

    token_ids = tokenizer.apply_chat_template(
        conversation,
        tokenize=True,
        add_generation_prompt=False,
    )

    return len(token_ids)


token_lengths = []

for index, example in enumerate(
    expanded_examples,
    start=1,
):
    token_lengths.append(
        full_conversation_token_length(
            example
        )
    )

    if index % 25 == 0:
        print(
            f"Checked {index} of "
            f"{len(expanded_examples)} examples..."
        )


token_summary = pd.Series(
    token_lengths
).describe()

display(token_summary)

print(
    "Examples above 512:",
    sum(
        length > MAX_SEQUENCE_LENGTH
        for length in token_lengths
    ),
)

Checked 25 of 120 examples...
Checked 50 of 120 examples...
Checked 75 of 120 examples...
Checked 100 of 120 examples...


,0
count,120.0
mean,2.0
std,0.0
min,2.0
25%,2.0
50%,2.0
75%,2.0
max,2.0


Examples above 512: 0


#### Full Fine-Tuning - Round Two

I'm keeping the training settings almost identical to the successful Milestone 3 run which keeps the training setup consistent while I test whether the larger 1.5B model handles the same expanded dataset and holdouts better.

In [10]:
model.config.use_cache = False

training_config = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    warmup_steps=4,
    weight_decay=0.01,
    logging_steps=2,
    eval_strategy="epoch",
    save_strategy="no",
    max_length=MAX_SEQUENCE_LENGTH,
    completion_only_loss=True,
    bf16=BF16_SUPPORTED,
    fp16=not BF16_SUPPORTED,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },
    optim=OPTIMIZER,
    report_to="none",
    dataloader_pin_memory=True,
    dataloader_num_workers=2,
    packing=False,
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=training_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

print("Trainer created.")
print("Output directory:", OUTPUT_DIR)
print("Optimizer:", OPTIMIZER)
print("Intermediate checkpoint saving: off")


/tmp/ipykernel_829/939941534.py:3: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  training_config = SFTConfig(


Tokenizing train dataset:   0%|          | 0/107 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/13 [00:00<?, ? examples/s]

Trainer created.
Output directory: /content/qwen-nba-1.5b-full-sft-v1
Optimizer: paged_adamw_8bit
Intermediate checkpoint saving: off


In [11]:
training_started = time.perf_counter()

training_result = trainer.train()

training_wall_seconds = (
    time.perf_counter()
    - training_started
)

evaluation_metrics = trainer.evaluate()

print("\nTraining completed.")
print(
    "Training time:",
    f"{training_wall_seconds / 60:.2f} minutes",
)
print("Training loss:", training_result.metrics.get("train_loss"))
print("Eval loss:", evaluation_metrics.get("eval_loss"))
print("Global steps:", trainer.state.global_step)
print("Best of the best of the best checkpoint:", trainer.state.best_model_checkpoint)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,0.448138,0.546718,0.535721,0.865509,39506.000000
2,0.278875,0.454839,0.427276,0.892923,79012.000000
3,0.105423,0.445787,0.347784,0.896232,118518.000000


Training Loss,Validation Loss,Epoch,Entropy,Mean Token Accuracy,Num Tokens
0.105423,0.445787,3,0.347784,0.896232,118518.000000



Training completed.
Training time: 12.52 minutes
Training loss: 0.44465697841879764
Eval loss: 0.44578665494918823
Global steps: 81
Best of the best of the best checkpoint: None


##### Saving the Improved Model
No extra comments really needed, pretty self explanatory

In [12]:
trainer.save_model(
    str(OUTPUT_DIR)
)

tokenizer.save_pretrained(
    str(OUTPUT_DIR)
)

saved_metrics = {
    "model_name": MODEL_NAME,
    "output_directory": str(OUTPUT_DIR),
    "full_fine_tuning": True,
    "lora_used": False,
    "total_parameters": total_parameters,
    "trainable_parameters": trainable_parameters,
    "training_examples": len(train_dataset),
    "evaluation_examples": len(eval_dataset),
    "training_wall_seconds": training_wall_seconds,
    "training_metrics": training_result.metrics,
    "evaluation_metrics": evaluation_metrics,
    "best_checkpoint": trainer.state.best_model_checkpoint,
}

with METRICS_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        saved_metrics,
        file,
        indent=2,
        default=str,
    )

print("New&Improved model:", OUTPUT_DIR)
print("Metrics:", METRICS_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

New&Improved model: /content/qwen-nba-1.5b-full-sft-v1
Metrics: /content/nba_1.5b_full_sft_metrics.json


##### Downloading the Colab Output

I'm packaging the trained model and result files into one ZIP so I can download the completed run back to my computer.


In [14]:
import zipfile

archive_path = PROJECT_DIR / "nba_1.5b_colab_run.zip"

print("Creating model archive...")

with zipfile.ZipFile(
    archive_path,
    "w",
    compression=zipfile.ZIP_STORED,
) as archive:

    #Add the full trained model directory
    for file_path in OUTPUT_DIR.rglob("*"):
        if file_path.is_file():
            archive.write(
                file_path,
                arcname=(
                    Path(OUTPUT_DIR.name)
                    / file_path.relative_to(OUTPUT_DIR)
                ),
            )

    #Add result files
    for result_file in [
        METRICS_PATH,
        HOLDOUT_OUTPUT_PATH,
        EXPANDED_DATASET_PATH,
    ]:
        if result_file.exists():
            archive.write(
                result_file,
                arcname=result_file.name,
            )

print(
    "Archive created:",
    archive_path,
)

print(
    "Archive size:",
    round(
        archive_path.stat().st_size
        / (1024 ** 3),
        2,
    ),
    "GB",
)

print("Starting browser download...")

files.download(
    str(archive_path)
)

Creating model archive...
Archive created: /content/nba_1.5b_colab_run.zip
Archive size: 2.89 GB
Starting browser download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#### Reusing the Milestone 3 Holdouts

I'm not inventing a brand-new test just because I changed the training data. The whole point is to see whether the exact failure cases from Milestone 3 improve - especially Ja/Jaylen (basically the hallucinations/made up players), the three-team Lillard trade, and that absolutely cursed Anthony Davis hypothetical... which i still have mixed feelings about.

In [15]:
holdout_tests = [
    {
        "id": 1,
        "category": "game_score_only",
        "prompt": (
            "Warriors Gold beat the Lakers 104-72 in the California Classic - "
            "do you have any more detail to add? Give me a short game recap "
            "and explain what probably mattered most."
        ),
    },
    {
        "id": 2,
        "category": "game_score_only",
        "prompt": (
            "Miami beat San Antonio 88-87 in the California Classic. What happened "
            "in the game and what were pivotal moments? Write an analytical recap."
        ),
    },
    {
        "id": 3,
        "category": "trade_vague",
        "prompt": (
            "Where did Ja get traded to and why? Who won this trade? Give each team "
            "a tentative grade and explain what information is missing."
        ),
    },
    {
        "id": 4,
        "category": "trade_vague",
        "prompt": (
            "I saw Jaylen Brown got traded, what happened? Who won this trade? Give "
            "both teams a tentative grade and explain what context would make the "
            "answer stronger."
        ),
    },
    {
        "id": 5,
        "category": "offseason_holdout",
        "prompt": (
            "What are the Lakers trying to accomplish this offseason? I saw LeBron "
            "may be leaving and they added Walker Kessler. Give me the big-picture "
            "strategy read and a tentative offseason grade."
        ),
    },
    {
        "id": 6,
        "category": "game_structured",
        "prompt": (
            "Charlotte Hornets beat Minnesota Timberwolves 110-108 on 2022-11-25. "
            "Charlotte Hornets shot 44.9% from the field compared with 42.7% for "
            "Minnesota Timberwolves. Charlotte Hornets made 8 three-pointers compared "
            "with 9. the rebounding totals were 55-45. the assist totals were 24-24. "
            "Charlotte Hornets committed 18 turnovers compared with 16. Charlotte "
            "Hornets outscored Minnesota Timberwolves 39-21 in the third quarter. "
            "points in the paint were 60-54. second-chance points were 14-15. Write "
            "an analytical recap explaining what likely mattered most, using only "
            "these numbers."
        ),
    },
    {
        "id": 7,
        "category": "game_structured",
        "prompt": (
            "Orlando Magic beat Miami Heat 105-90 on 2018-12-04. Orlando Magic shot "
            "44.2% from the field compared with 41.8% for Miami Heat. Orlando Magic "
            "made 14 three-pointers compared with 12. the rebounding totals were 43-47. "
            "the assist totals were 22-19. Orlando Magic committed 15 turnovers compared "
            "with 16. Write an analytical recap explaining what likely mattered most, "
            "using only these numbers."
        ),
    },
    {
        "id": 8,
        "category": "game_structured",
        "prompt": (
            "Orlando Magic beat Dallas Mavericks 111-106 on 2019-03-08. Orlando Magic "
            "shot 48.3% from the field compared with 47.2% for Dallas Mavericks. Orlando "
            "Magic made 14 three-pointers compared with 10. the rebounding totals were "
            "48-42. the assist totals were 26-22. Orlando Magic committed 16 turnovers "
            "compared with 14. Orlando Magic outscored Dallas Mavericks 33-19 in the "
            "fourth quarter. points in the paint were 56-42. second-chance points were "
            "4-14. Write an analytical recap explaining what likely mattered most, using "
            "only these numbers."
        ),
    },
    {
        "id": 9,
        "category": "trade_structured",
        "prompt": (
            "Milwaukee acquired Damian Lillard in a three-team trade. Portland received "
            "Jrue Holiday, Deandre Ayton, Toumani Camara, a first-round pick, and two "
            "pick swaps, while Phoenix received Grayson Allen, Jusuf Nurkic, Nassir "
            "Little, and Keon Johnson. Give each team's side a short tentative grade."
        ),
    },
    {
        "id": 10,
        "category": "trade_potential",
        "prompt": (
            "Potential trade idea: Oklahoma City receives Anthony Davis from the Lakers "
            "for one young starter and three first-round picks. Treat this only as a "
            "hypothetical and grade both teams."
        ),
    },
]

In [16]:
def generate_response(
    current_model,
    user_message,
    max_new_tokens=260,
):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_message,
        },
    ]

    model_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    model_inputs = model_inputs.to(
        next(
            current_model.parameters()
        ).device
    )

    prompt_length = (
        model_inputs["input_ids"]
        .shape[-1]
    )

    with torch.no_grad():
        output_ids = current_model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    response_ids = output_ids[
        0,
        prompt_length:,
    ]

    return tokenizer.decode(
        response_ids,
        skip_special_tokens=True,
    ).strip()


improved_model = trainer.model
improved_model.gradient_checkpointing_disable()
improved_model.config.use_cache = True
improved_model.eval()

improved_holdout_results = {}

for test in holdout_tests:
    started = time.perf_counter()

    response = generate_response(
        improved_model,
        test["prompt"],
    )

    elapsed = (
        time.perf_counter()
        - started
    )

    improved_holdout_results[
        test["id"]
    ] = {
        "category": test["category"],
        "prompt": test["prompt"],
        "response": response,
        "seconds": elapsed,
    }

    print("=" * 110)
    print(
        f"Holdout {test['id']} | "
        f"{test['category']}"
    )
    print("Prompt:")
    print(test["prompt"])
    print("\nNew&Improved model reply:")
    print(response)
    print(
        f"\nReply time: "
        f"{elapsed:.2f} seconds"
    )
    print()

with HOLDOUT_OUTPUT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        improved_holdout_results,
        file,
        indent=2,
        ensure_ascii=False,
    )

print(
    "Holdout outputs saved to:",
    HOLDOUT_OUTPUT_PATH,
)

Holdout 1 | game_score_only
Prompt:
Warriors Gold beat the Lakers 104-72 in the California Classic - do you have any more detail to add? Give me a short game recap and explain what probably mattered most.

New&Improved model reply:
Warriors Gold 104, Lakers 72: The clearest supplied details were that the Warriors beat the Lakers in the California Classic. Based only on that information, I can't responsibly identify specific players, gamescores, or highlight plays. A clearer trade situation or official box score would help me grade this matchup.

Reply time: 3.62 seconds

Holdout 2 | game_score_only
Prompt:
Miami beat San Antonio 88-87 in the California Classic. What happened in the game and what were pivotal moments? Write an analytical recap.

New&Improved model reply:
I don't have enough information to say for sure what happened in this game, because I can't retain the exact teams, full score, individual performances, or top stats. A recap should always include those specifics. Is th

#### Sample RAG Demo - Moved to net new notebook!

#### Iteration Notes

This notebook is intentionally smaller than Milestone 3 because I'm not re-documenting the entire project from scratch. The main changes are the expanded trade training data, the larger 1.5B model, and the move to a Google Colab CUDA GPU for this full fine-tuning run. Once the holdouts run, the important question is pretty simple: did the trade replies stop inventing names/packages and did the Anthony Davis hypothetical finally make basic sense?

The Streamlit application uses the same ideas in a more practical way: optional structured game facts can be calculated in Python before generation, player names can be validated against the SQLite database, and the retrieved player snapshot can be passed to the fine-tuned model when the Sample RAG Demo is enabled.

Also, ignore the RAG demo - I split it into a different notebook due to memory issues, as I don't want to continously going back and forth reloading this particular notebook.